In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors.Punched import PunchedCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.MonoLayers import ZFoldMonoLayer
from steer_opencell_design.Constructions.ElectrodeAssemblies.Stacks import ZFoldStack

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data('cells', "name == 'Vendor D NFPP'")

In [4]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Vendor E NFPP,gASVIQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-12 10:16:06,0.4.1,Na/Na+
1,Vendor G NFM,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 10:16:08,0.4.1,Na/Na+
2,Vendor G NFPP,gASV7BEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 10:16:10,0.4.1,Na/Na+
3,CBAK-32140NS,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:01,0.4.1,Na/Na+
4,Cell 2,gASV7w4BAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:03,0.4.1,Na/Na+
5,Cell 4,gASVFQkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:06,0.4.1,Na/Na+
6,HiNa-NaCR32140-MP10,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:08,0.4.1,Na/Na+
7,QNAS-S40160NL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:10,0.4.1,Na/Na+
8,QNAS-S40160RL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:13,0.4.1,Na/Na+
9,UniGrid-NCO32140,gASVLwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:15,0.4.1,Na/Na+


In [5]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials']

In [6]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polyethylene")


In [7]:
# Create the cathode

cathode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=166,
    height=189,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=50,
    coated_tab_height=4,
    insulation_width=5
)

cathode_active_material = CathodeMaterial.from_database("NFPP")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=1.88,
    mass_loading=18.12,
    insulation_material=insulation_material,
    insulation_thickness=3,
)



In [8]:
# Create the anode object

anode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=168,
    height=191,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=118,
    coated_tab_height=4,
    insulation_width=5
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=9,
    insulation_material=insulation_material,
    insulation_thickness=3
)



In [9]:
# make the layup

separator = Separator(
    material=separator_material,
    width=192,
    thickness=25
)

my_monolayer = ZFoldMonoLayer(
    anode=my_anode,
    cathode=my_cathode,
    separator=separator
)


In [10]:
# make the stack

my_stack = ZFoldStack(
    layup=my_monolayer,
    n_layers=40,
    name="Vendor D NFPP",
    additional_separator_wraps=6
)

In [11]:
print(f"Stack cost: {my_stack.cost:.2f} USD")
print(f"Stack mass: {my_stack.mass:.2f} g")

Stack cost: 9.75 USD
Stack mass: 874.22 g


In [12]:
my_stack.get_side_view(width=1600, height=500).show()

In [13]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_stack.name],
    'object': [my_stack.serialize()],
    'form_factor': ['Prismatic'],
    'internal_construction': ['Stacked'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [14]:
db.get_data(table_name='cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Vendor E NFPP,gASVIQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-12 10:16:06,0.4.1,Na/Na+
1,Vendor G NFM,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 10:16:08,0.4.1,Na/Na+
2,Vendor G NFPP,gASV7BEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 10:16:10,0.4.1,Na/Na+
3,CBAK-32140NS,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:01,0.4.1,Na/Na+
4,Cell 2,gASV7w4BAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:03,0.4.1,Na/Na+
5,Cell 4,gASVFQkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:06,0.4.1,Na/Na+
6,HiNa-NaCR32140-MP10,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:08,0.4.1,Na/Na+
7,QNAS-S40160NL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:10,0.4.1,Na/Na+
8,QNAS-S40160RL,gASVmwwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:13,0.4.1,Na/Na+
9,UniGrid-NCO32140,gASVLwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-12 11:38:15,0.4.1,Na/Na+
